In [1]:
import json

with open("data/products.json", encoding="utf-8") as file:
    products = json.load(file) # reads json file and converts it to python

# get description from a product
descriptions = [product["description"] for product in products] 
len(descriptions)

10

In [2]:
from pydantic_ai import Agent
from model import nvidia_model

model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b")

In [3]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="Write the name, brand and model of the product")
    price: float = Field(description="Write the price of the product")
    category: str = Field(description="What category it is")
    in_stock: bool = Field(description="If it's in stock, then True, else False")
    description: str = Field(description="Describe the product in on sentance")

product_agent = Agent(
    model=model,
    instructions="You extract product information from short product descriptions. Only use information from the text",
)

async def extract_product(text: str) -> Product:
    result = await product_agent.run(text, output_type=Product)
    return result.output 


In [4]:
import pandas as pd

extracted = []
for text in descriptions:
    product = await extract_product(text)
    extracted.append(product)

df_extracted = pd.DataFrame([p.model_dump() for p in extracted])
df_extracted
df_facit = pd.DataFrame(products)

In [8]:
class Judgement(BaseModel):
    same_meaning: bool = Field(description="True if both texts describe the same product with the same facts")
    reason: str = Field(description="Short explanation, one sentence")

judge_agent = Agent(
    model=model,
    instructions=(
        "You compare two product descriptions. "
        "Decide if they describe the same product with the same facts. "
        "The wording may differ, only the facts matter."
    ),
)

async def judge(original: str, extracted: str) -> Judgement:
    prompt = f"Original:\n{original}\n\nExtracted:\n{extracted}"
    result = await judge_agent.run(prompt, output_type=Judgement)
    return result.output

In [9]:
judgements = []
for original, product in zip(descriptions, extracted):
    judgement = await judge(original, product.description)
    judgements.append(judgement)

df_extracted["description_ok"] = [j.same_meaning for j in judgements]
df_extracted["description_ok"].sum()

np.int64(6)

In [5]:
df_facit[["name", "price", "category", "in_stock"]]

,name,price,category,in_stock
0,Nike Running Shoes X90,120.00,Footwear,True
1,Sony WH-1000XM5 Headphones,349.99,Electronics,True
2,Dyson V12 Vacuum Cleaner,599.99,Home & Garden,False
3,Adidas Ultraboost 22 Sneakers,189.95,Footwear,True
4,Apple iPad Pro 11-inch,799.00,Electronics,True
5,Levi's 501 Original Jeans,69.50,Clothing,True
6,Bosch Cordless Drill GSR18,149.00,Automotive,False
7,Canon EOS R50 Camera,679.99,Electronics,True
8,Philips Air Purifier 3000i,299.95,Home & Garden,True
9,Samsung Galaxy Watch 6,299.00,Electronics,False


In [6]:
df_extracted[["name", "price", "category", "in_stock"]]

,name,price,category,in_stock
0,Nike Running Shoes X90,120.00,Footwear,True
1,Sony WH-1000XM5 Headphones,349.99,Electronics,True
2,Dyson V12 Vacuum Cleaner,599.99,Home & Garden,False
3,Adidas Ultraboost 22 Sneakers,189.95,Footwear,True
4,Apple iPad Pro 11-inch,799.00,Electronics,True
5,Levi's 501 Original Jeans,69.50,Clothing,True
6,Bosch Cordless Drill GSR18,149.00,Automotive,False
7,Canon EOS R50 Camera,679.99,Electronics,True
8,Philips Air Purifier 3000i,299.95,Home & Garden,True
9,Samsung Galaxy Watch 6,299.00,Electronics,False


In [7]:
for column in ["name", "price", "category", "in_stock"]:
    correct = (df_facit[column] == df_extracted[column]).sum()
    print(column, correct, "av", len(df_facit))

name 10 av 10
price 10 av 10
category 10 av 10
in_stock 10 av 10
